In [2]:
import numpy as np
import matplotlib as plt
import seaborn as sea
import pandas as pd
import xgboost as xgb
import sklearn as sk
import sqlite3 
import mysql.connector
import duckdb # for me to use sql even though it is not necessary with the way the datasets are given

## Data Definition

I am asked by Northern Lights Air (NLA) to run an analysis on their loyalty program. NLA ran a promotional campaign from Februrary 2018 (2018-02-01) until April 2018 (2018-04-31). NLA wants to improve their loyalty program, hence they want to know:
- What impact did the campaign have on the loyalty memberships? (Enrollment, Cancellation, Points redeemed etc.)
- What impact did the campaign have on booked flights? (in the summer, in general etc.)
- What impact did it have on customer churning?
- Was the campaign more successful towards certain demographics? (Gender, Education, Marital Status etc.)

## Exploratory Analysis

I will use SQL (for fun) to explore the data. 

**392936** total data entries index starting at 0

### Calendar 
This is a dataset that gives me everyday from 2012-01-01 (January 1, 2012) until 2018-12-31 (December 31, 2018). It gives me:
- Start of Year: first date of the year
- Start of Quarter: first date of the quarter it's in (Jan 1, Apr 1, Jul 1, Oct 1)
- Start of Month: first date of the month it's in

### Customer Flights Activity 
- Loyalty Number: **16737** unique members based on loyalty numbers
- Year
- Month
- Total Flights: Do not have to be loyalty member to fly
- Distance
- Points Accumulated
- Points Redeemed
- Dollar Cost Points Redeemed (CAD)

### Loyalty Program 
- Loyalty Number: **16737** unique members based on loyalty numbers
- Country
- Province
- City
- Postal Code
- Gender
- Education
- Salary (CAD)
- Marital Status
- Loyalty Card
- CLV
- Enrollment Type
- Enrollment Year
- Enrollment Month
- Cancellation Year
- Cancellation Month

### New Column Ideas
- Enrollment_length: Amount of time you've been in the program

I can see that we can **combine** Enrollment Year and Month together into one column, as well as Cancellation Year and Month. \
Can **combine** the two tables via Loyalty Number. \
Make two different tables, one that matches enrolment date to the calendar and one that matches cancellation date to calendar.\
Notice that column names are strings let's change that


In [19]:
#Quick check to see the tables
calendar_df = duckdb.sql("SELECT * FROM read_csv_auto('Calendar.csv')").df()
flight_df = duckdb.sql("SELECT * FROM read_csv_auto('Customer Flight Activity.csv')").df()
loyalty_df = duckdb.sql("SELECT * FROM read_csv_auto('Customer Loyalty History.csv')").df()

# print(calendar_df)
# print(flight_df)
# print(loyalty_df)

NameError: name 'read_csv_auto' is not defined

### Checking our data

In [49]:
#Concat Month and Year togetherabs
duckdb.sql(""" 
    SELECT 
        MAKE_DATE(Year, Month, 1) AS "Flight Date",
        MAKE_DATE("Enrollment Year", "Enrollment Month", 1) AS "Enrollment Date",
        MAKE_DATE("Cancellation Year", "Cancellation Month", 1) AS "Cancellation Date"
    FROM read_csv_auto('Customer Flight Activity.csv') cfa
    LEFT JOIN read_csv_auto('Customer Loyalty History.csv') clh
        on cfa."Loyalty Number" = clh."Loyalty Number"
           """).df()


,Flight Date,Enrollment Date,Cancellation Date
0,2017-11-01,2015-01-01,NaT
1,2017-11-01,2016-04-01,NaT
2,2017-11-01,2017-12-01,NaT
3,2017-10-01,2014-06-01,2018-09-01
4,2017-11-01,2014-06-01,2018-09-01
...,...,...,...
392931,2017-11-01,2013-03-01,NaT
392932,2017-11-01,2016-08-01,NaT
392933,2017-11-01,2015-03-01,NaT
392934,2017-01-01,2012-11-01,NaT


In [28]:
#double check that you don't have to be a loyalty member to fly
duckdb.sql(""" 
    SELECT 
        cfa."Loyalty Number",
        Year,
        "Cancellation Year"
    FROM read_csv_auto('Customer Flight Activity.csv') cfa
    LEFT JOIN read_csv_auto('Customer Loyalty History.csv') clh
        on cfa."Loyalty Number" = clh."Loyalty Number"
    WHERE "Cancellation Year" < "Year"
           """).df()

,Loyalty Number,Year,Cancellation Year
0,427804,2017,2016
1,428872,2017,2015
2,429689,2017,2014
3,430519,2017,2016
4,430559,2017,2016
...,...,...,...
28315,421432,2017,2016
28316,421855,2017,2016
28317,423313,2017,2016
28318,424027,2017,2014


In [27]:
#Joining the tables via SQL

duckdb.sql("""
    SELECT *
    FROM read_csv_auto('Customer Flight Activity.csv') cfa
    LEFT JOIN read_csv_auto('Customer Loyalty History.csv') clh
        on cfa."Loyalty Number" = clh."Loyalty Number"
           """).df()


,Loyalty Number,Year,Cancellation Year
0,427804,2017,2016
1,428872,2017,2015
2,429689,2017,2014
3,430519,2017,2016
4,430559,2017,2016
...,...,...,...
28315,421432,2017,2016
28316,421855,2017,2016
28317,423313,2017,2016
28318,424027,2017,2014
